# Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

- A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
- A function or coroutine to execute.

In [9]:

import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "Why do parrots talk?" This is a straightforward question about animal behavior, specifically avian vocal learning in parrots.\n\n2.  **Identify Key Concepts**:\n   - Parrots are known for mimicking human speech\n   - This ability is called vocal learning\n   - It\'s not unique to parrots (also seen in songbirds, hummingbirds, dolphins, whales, elephants, bats, humans)\n   - Evolutionary reasons: social bonding, communication within flocks, survival, intelligence\n   - Biological mechanisms: specialized brain structures (song system), syrinx anatomy, neural plasticity\n   - Context: In the wild vs. captivity\n\n3.  **Structure the Answer**:\n   - Introduction: Clarify that parrots don\'t "talk" in the human sense but mimic sounds\n   - Biological capability: Vocal learning ability, brain structure, syrinx\n   - Evolutionary/social reasons: Why they evolved this trait\n   - Wild 

In [10]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather in a given location."""
    return f"The weather in {location} is sunny."

In [11]:
model_with_tools = model.bind_tools([get_weather])

In [12]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify user intent: User wants to know the current weather in Boston.\n2.  Identify available tools: `get_weather` function takes a `location` parameter.\n3.  Extract parameters: `location` = "Boston".\n4.  Call the tool: `get_weather(location="Boston")`.\n5.  Formulate response based on the tool output. (I will simulate the tool call now)✅\n', 'tool_calls': [{'id': 'qq1aebpvm', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 278, 'total_tokens': 401, 'completion_time': 0.245528557, 'completion_tokens_details': {'reasoning_tokens': 95}, 'prompt_time': 0.02424777, 'prompt_tokens_details': None, 'queue_time': 0.252215765, 'total_time': 0.269776327}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_b545673577', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logpro

In [13]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


## Tool Execution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Longueuil?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Longueuil is 24°C and sunny."

The weather in Longueuil is sunny.


In [21]:
messages

[{'role': 'user', 'content': "What's the weather in Longueuil?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s request: User wants to know the weather in Longueuil.\n2.  Identify the location: Longueuil.\n3.  Check available tools: `get_weather` function takes a `location` parameter.\n4.  Call the tool: `get_weather(location="Longueuil")`.\n5.  Process the response and formulate the answer. (Wait for tool output) -> I will simulate the tool call. Since I am an AI, I will actually call the tool.\nLet\'s call the tool. \nParameters: `{"location": "Longueuil"}`.\nProceed. \n(Self-Correction/Verification during thought)\nThe tool definition is clear. I will generate the tool call.✅\nOutput matches the required format. \nProceeding. \n`get_weather(location="Longueuil")` -> calls the function.\nI will generate the tool call now. \n[Note: I am acting as the model that generates tool calls. I will output the tool call exactly as sp